# Install & Import

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session


session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
STAGE = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"

In [ ]:
TARGET_COLS = [
    'Electrical Conductance',
    'Total Alkalinity',
    'Dissolved Reactive Phosphorus'
]
KEY_COLS_FULL = ['Latitude', 'Longitude', 'Sample_Date', 'Sample Date']

SANLC_IMPACT_MAP = {
    'mines: extraction pits, quarries': 4,
    'commercial': 3,
    'commercial annual crops pivot irrigated': 3,
    'commercial annual crops rain-fed / dryland': 3,
    'cultivated commercial permanent orchards': 3,
    'cultivated commercial permanent vines': 3,
    'cultivated commercial sugarcane non-pivot': 3,
    'subsistence / small-scale annual crops': 2,
    'residential formal (low veg / grass)': 2,
    'smallholdings (low veg / grass)': 2,
    'fallow land & old fields (grass)': 1,
    'fallow land & old fields (trees)': 1,
    'other bare': 1,
    'natural grassland': 0,
    'dense forest & woodland': 0,
    'open woodland': 0,
    'contiguous (indigenous) forest': 0,
    'contiguous & dense plantation forest': 0,
    'contiguous low forest & thicket': 0,
    'low shrubland (fynbos)': 0,
    'low shrubland (nama karoo)': 0,
    'low shrubland (succulent karoo)': 0,
    'herbaceous wetlands (previously mapped)': 0,
    'Unclassified/No Data': -1,
}
SANLC_TEXT_COLS = ['SANLC2022_CLASS_1KM', 'SANLC2020_CLASS_1KM']
# --- Toggle flags ---
APPLY_LOG_TRANSFORM = False  # Set True di iterasi 2 jika DRP skew>1

# --- Reusable helpers ---
def load_from_stage(filename):
    local_path = f"/tmp/{filename}"
    if not os.path.exists(local_path):
        session.file.get(f"{STAGE}/{filename}", "/tmp/")
    if filename.endswith('.parquet'):
        return pd.read_parquet(local_path)
    return pd.read_csv(local_path)

def get_feature_cols(df):
    exclude = set(KEY_COLS_FULL + TARGET_COLS)
    return [c for c in df.columns if c not in exclude]

def encode_sanlc_column(df, col):
    """Encode 1 kolom SANLC text → numeric impact score."""
    # Lowercase dulu isi teksnya untuk matching yang konsisten
    mapped = df[col].str.lower().str.strip().map(SANLC_IMPACT_MAP)

    # Cek apakah ada kelas yang TIDAK terdaftar di map
    unmapped_mask = mapped.isna() & df[col].notna()
    if unmapped_mask.any():
        unmapped_classes = df.loc[unmapped_mask, col].unique()
        print(f"  ⚠ Kelas TIDAK DIKENALI di {col}: {unmapped_classes}")
        print(f"    → {unmapped_mask.sum()} baris akan di-set ke -1 (unknown)")

    new_col = col.lower().replace('_class_', '_impact_')
    df[new_col] = mapped.fillna(-1).astype(np.int8)
    df = df.drop(columns=[col])
    return df

def upload_to_stage(df, filename):
    path = f"/tmp/{filename}"
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    session.file.put(f"file://{path}", STAGE, auto_compress=False, overwrite=True)
    mb = os.path.getsize(path) / 1024 / 1024
    print(f"✓ {filename}: {df.shape} → {mb:.2f} MB")

def assign_region(row):
    lat, lon = row['Latitude'], row['Longitude']
    if lat <= -32.0 and lon < 24.0:
        return 'Western_Cape'
    elif lat >= -29.0:
        return 'Northern_Bulk'
    elif lat < -29.0 and lon >= 24.0:
        return 'Eastern_Cape'
    else:
        return 'Karoo_Buffer'

print("Setup done")

# Data Ingestion

In [ ]:
target_train = load_from_stage("water_quality_training_dataset.csv")
target_test = load_from_stage("submission_template.csv")

for df in [target_train, target_test]:
    df['Sample_Date'] = pd.to_datetime(df['Sample Date'], format='mixed', dayfirst=False)

data_map = {
    'landsat_train': 'landsat_features_training.csv',
    'landsat_test':  'landsat_features_validation.csv',
    'terra_train':   'terraclimate_features_training.csv',
    'terra_test':    'terraclimate_features_validation.csv',
}
ey_data = {k: load_from_stage(v) for k, v in data_map.items()}

ext_files = {
    'elevation':  ('elevation_features.parquet',       'test_elevation_features.parquet'),
    'weather':    ('weather_features.parquet',          'test_weather_features.parquet'),
    'soilgrids':  ('soilgrids_features.parquet',        'test_soilgrids_features.parquet'),
    'sanlc':      ('train_sanlc_features.parquet',      'test_sanlc_features.parquet'),
    'worldpop':   ('train_worldpop_features.parquet',   'test_worldpop_features.parquet'),
    'hydro':      ('train_hydro_features.parquet',      'test_hydro_features.parquet'),
}
ext_train = {}
ext_test = {}
for name, (f_train, f_test) in ext_files.items():
    try:
        ext_train[name] = load_from_stage(f_train)
        ext_test[name] = load_from_stage(f_test)
        print(f"  ✓ {name}: train={ext_train[name].shape}, test={ext_test[name].shape}")
    except Exception as e:
        print(f"  ⚠ {name}: SKIP — {e}")

print(f"\n Target train: {target_train.shape}, test: {target_test.shape}")

## Encode

In [ ]:
for split_name, split_dict in [('train', ext_train), ('test', ext_test)]:
    if 'sanlc' in split_dict:
        df = split_dict['sanlc']
        
        # Debug: print kolom yang sebenarnya ada
        print(f"\n  [{split_name}] Kolom SANLC sebelum encode: {df.columns.tolist()}")
        print(f"  [{split_name}] Sample values:")
        for col in df.columns:
            if 'class' in col.lower() or 'sanlc' in col.lower():
                print(f"    {col}: {df[col].dropna().unique()[:5]}")
        
        # Encode setiap kolom text SANLC
        for col in SANLC_TEXT_COLS:
            if col in df.columns:
                print(f"  → Encoding {col}...")
                df = encode_sanlc_column(df, col)
            else:
                # Coba cari case-insensitive
                matched = [c for c in df.columns if c.lower() == col.lower()]
                if matched:
                    actual_col = matched[0]
                    print(f"  → Encoding {actual_col} (matched case-insensitive)...")
                    df = encode_sanlc_column(df, actual_col)
                else:
                    print(f"  ⚠ Kolom {col} TIDAK ditemukan di {split_name}")
        
        split_dict['sanlc'] = df
        print(f"  ✓ SANLC {split_name} encoded: {df.columns.tolist()}")

# SANLC change detection: apakah lahan berubah antara 2020-2022
for split_dict in [ext_train, ext_test]:
    if 'sanlc' in split_dict:
        df = split_dict['sanlc']
        impact_2022 = [c for c in df.columns if '2022' in c and 'impact' in c]
        impact_2020 = [c for c in df.columns if '2020' in c and 'impact' in c]
        
        if impact_2022 and impact_2020:
            df['sanlc_change_2020_2022'] = (
                df[impact_2022[0]] - df[impact_2020[0]]
            ).astype(np.int8)
            print(f"  ✓ sanlc_change_2020_2022 created")
        
        split_dict['sanlc'] = df

print("\n✓ SANLC encoding selesai")

### Leave One Region Spliy

In [ ]:
target_train['Region'] = target_train.apply(assign_region, axis=1)

print("Data distribution across regions:")
print(target_train['Region'].value_counts())

# Hitung unique locations per region berdasarkan koordinat (bukan station number)
print(f"\nUnique locations per region:")
print(
    target_train.groupby('Region')[['Latitude', 'Longitude']]
    .apply(lambda g: g.drop_duplicates().shape[0])
    .rename('unique_locations')
)

In [ ]:
target_train.info()

## Merging

In [ ]:
def master_merge(target_df, ey_dict, ext_dict, split='train'):
    df = target_df.copy()
    
    for ey_name in ['landsat', 'terra']:
        ey_key = f'{ey_name}_{split}'
        if ey_key not in ey_dict:
            continue
        ey_df = ey_dict[ey_key]
        feat_cols = get_feature_cols(ey_df)
        if not feat_cols:
            continue

        has_keys = 'Latitude' in ey_df.columns and 'Longitude' in ey_df.columns
        if has_keys and len(ey_df) == len(df):
            if 'Sample Date' in ey_df.columns and 'Sample_Date' not in ey_df.columns:
                ey_df['Sample_Date'] = pd.to_datetime(ey_df['Sample Date'], format='mixed', dayfirst=False)
            merge_keys = ['Latitude', 'Longitude']
            if 'Sample_Date' in ey_df.columns:
                merge_keys.append('Sample_Date')
            df = df.merge(
                ey_df[merge_keys + feat_cols].drop_duplicates(subset=merge_keys),
                on=merge_keys, how='left', suffixes=('', f'_{ey_name}')
            )
        else:
            df = pd.concat([df.reset_index(drop=True),
                           ey_df[feat_cols].reset_index(drop=True)], axis=1)
        print(f"  + {ey_name}: +{len(feat_cols)} features → {df.shape}")
    
    for name, ext_df in ext_dict.items():
        feat_cols = get_feature_cols(ext_df)
        if not feat_cols:
            continue
        has_date = 'Sample_Date' in ext_df.columns
        merge_keys = [c for c in ['Latitude', 'Longitude'] if c in ext_df.columns]
        if has_date:
            merge_keys.append('Sample_Date')
        if merge_keys:
            merge_df = ext_df[merge_keys + feat_cols].drop_duplicates(subset=merge_keys)
            df = df.merge(merge_df, on=merge_keys, how='left', suffixes=('', f'_{name}'))
        else:
            df = pd.concat([df.reset_index(drop=True),
                           ext_df[feat_cols].reset_index(drop=True)], axis=1)
        print(f"  + {name}: +{len(feat_cols)} features → {df.shape}")
    
    return df

print("=" * 60)
print("MASTER MERGE — Training Data")
print("=" * 60)
master_train = master_merge(target_train, ey_data, ext_train, 'train')

print(f"\n{'=' * 60}")
print("MASTER MERGE — Test Data")
print("=" * 60)
master_test = master_merge(target_test, ey_data, ext_test, 'test')

print(f"\nFINAL — master_train: {master_train.shape}")
print(f"FINAL — master_test:  {master_test.shape}")

# Sanity Check's

In [ ]:
numeric_cols = master_train.select_dtypes(include=[np.number]).columns
master_train[numeric_cols] = master_train[numeric_cols].fillna(0)

test_numeric = numeric_cols.intersection(master_test.columns)
master_test[test_numeric] = master_test[test_numeric].fillna(0)

missing = master_train.isnull().sum()
has_missing = missing[missing > 0]
if len(has_missing) > 0:
    print("⚠ Kolom masih ada NaN:")
    print(has_missing)
else:
    print("✓ Semua numeric bersih — 0 missing values")

n_dup = master_train.duplicated().sum()
print(f"\n  Duplicate rows: {n_dup}")
print(f"  Train shape: {master_train.shape}")
print(f"  Test shape:  {master_test.shape}")

# Hitung fitur (exclude keys, targets, dan Region)
non_feature = set(KEY_COLS_FULL + TARGET_COLS + ['Region'])
feat_count = len([c for c in master_train.columns if c not in non_feature])
print(f"  Total feature columns: {feat_count}")
print(f"  Region column: {'Region' in master_train.columns}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
transform_recommendations = {}

for i, col in enumerate(TARGET_COLS):
    data = master_train[col].dropna()
    skew = data.skew()
    
    axes[i].hist(data, bins=50, edgecolor='black', alpha=0.7, color=colors[i])
    axes[i].set_title(f'{col}\nskew={skew:.2f}, median={data.median():.1f}', fontsize=10)
    axes[i].axvline(data.median(), color='red', linestyle='--', label='Median')
    axes[i].legend(fontsize=8)
    
    transform_recommendations[col] = 'log1p' if abs(skew) > 1 else 'none'
    symbol = '⚠ PERLU log-transform' if abs(skew) > 1 else '✓ OK'
    print(f"  {col}: skew={skew:.2f} → {symbol}")

plt.suptitle('Distribution of 3 Target Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nTransform recommendations: {transform_recommendations}")

In [ ]:
if APPLY_LOG_TRANSFORM:
    for col in TARGET_COLS:
        if transform_recommendations.get(col) == 'log1p':
            original_skew = master_train[col].skew()
            master_train[f'{col}_original'] = master_train[col].copy()
            master_train[col] = np.log1p(master_train[col])
            new_skew = master_train[col].skew()
            print(f"  ✓ {col}: skew {original_skew:.2f} → {new_skew:.2f} (log1p applied)")
            print(f"    ⚠ INGAT: saat prediksi harus np.expm1() untuk inverse!")
    print("\n⚠ Log-transform AKTIF — prediksi di notebook 05 harus di-inverse!")
else:
    print("ℹ Log-transform TIDAK aktif (iterasi 1 baseline)")
    print("  Set APPLY_LOG_TRANSFORM = True di Cell 2 untuk iterasi 2")

In [ ]:
print("Correlation to target:")
target_corr = master_train[TARGET_COLS].corr()
print(target_corr.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(target_corr, annot=True, cmap='coolwarm', center=0, ax=axes[0], fmt='.3f')
axes[0].set_title('Inter-Target Correlation')

all_numeric = master_train.select_dtypes(include=[np.number])
corr_matrix = all_numeric.corr()

top_features_per_target = {}
for target in TARGET_COLS:
    if target not in corr_matrix.columns:
        continue
    target_corrs = corr_matrix[target].drop(TARGET_COLS, errors='ignore').abs().sort_values(ascending=False).head(10)
    top_features_per_target[target] = target_corrs.index.tolist()
    print(f"\nTop 10 features for {target}:")
    for feat in target_corrs.index:
        print(f"  {feat}: {corr_matrix[target][feat]:.4f}")

all_top = list(set(f for feats in top_features_per_target.values() for f in feats))
if all_top:
    sub_corr = master_train[all_top + TARGET_COLS].corr()[TARGET_COLS].drop(TARGET_COLS, errors='ignore')
    sns.heatmap(sub_corr, annot=True, cmap='RdBu_r', center=0, ax=axes[1], fmt='.2f',
                yticklabels=[c[:25] for c in sub_corr.index])
    axes[1].set_title('Top Features vs Targets')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Scatter map per region
region_colors = {
    'Western_Cape': '#E91E63',
    'Northern_Bulk': '#2196F3', 
    'Eastern_Cape': '#4CAF50',
    'Karoo_Buffer': '#FF9800'
}
ax = axes[0]
for region, color in region_colors.items():
    mask = master_train['Region'] == region
    ax.scatter(master_train.loc[mask, 'Longitude'], 
               master_train.loc[mask, 'Latitude'],
               c=color, label=region, alpha=0.5, s=10)
ax.set_title('Sample Stations by Region')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(fontsize=7)

# Boxplot per target per region
for i, col in enumerate(TARGET_COLS):
    ax = axes[i + 1]
    data_to_plot = []
    labels = []
    for region in region_colors:
        data_to_plot.append(master_train.loc[master_train['Region'] == region, col].dropna())
        labels.append(region.replace('_', '\n'))
    ax.boxplot(data_to_plot, labels=labels)
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', labelsize=7)

plt.suptitle('Target Distribution per Region (for GroupKFold validation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Extract to Parquet

In [ ]:
import joblib
drop_cols = ['Sample Date', 'Sample_Date', 'Latitude', 'Longitude']

# Simpan Region sebelum drop koordinat
region_array = master_train['Region'].values.copy()

# Drop key columns + Region (Region bukan fitur, hanya untuk CV grouping)
train_export = master_train.drop(
    columns=[c for c in drop_cols if c in master_train.columns]
)
test_export = master_test.drop(
    columns=[c for c in drop_cols if c in master_test.columns]
)

# --- Upload parquet ---
upload_to_stage(train_export, "master_train.parquet")
upload_to_stage(test_export, "master_test.parquet")

# --- Simpan region sebagai joblib (untuk GroupKFold di notebook 04/05) ---
region_bundle = {
    'region_array': region_array,
    'region_names': list(master_train['Region'].unique()),
    'region_counts': master_train['Region'].value_counts().to_dict(),
    'n_samples': len(region_array),
}
joblib.dump(region_bundle, '/tmp/region_groups.joblib', compress=('zlib', 3))
session.file.put("file:///tmp/region_groups.joblib", STAGE, auto_compress=False, overwrite=True)
print(f"✓ region_groups.joblib: {len(region_array)} samples, {len(region_bundle['region_names'])} regions")

# --- Pipeline metadata ---
non_feature = set(KEY_COLS_FULL + TARGET_COLS + ['Region'])
feature_names = [c for c in train_export.columns if c not in TARGET_COLS]

meta = {
    'target_cols': TARGET_COLS,
    'n_features': len(feature_names),
    'n_train_rows': len(train_export),
    'n_test_rows': len(test_export),
    'transform_recommendations': transform_recommendations,
    'log_transform_applied': APPLY_LOG_TRANSFORM,
    'feature_names': feature_names,
    'regions': region_bundle['region_counts'],
}
with open('/tmp/pipeline_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
session.file.put("file:///tmp/pipeline_meta.json", STAGE, auto_compress=False, overwrite=True)

print(f"\n✓ Exported: {meta['n_features']} features, {meta['n_train_rows']} train rows")
print(f"  Regions: {meta['regions']}")

## Feature Engineering

In [ ]:
ENABLE_FEATURE_ENGINEERING = False  # Set True di iterasi 2

if ENABLE_FEATURE_ENGINEERING:
    print("Feature Engineering aktif — menambahkan fitur derivatif...")
    
    # --- Ratio features ---
    if 'NDMI' in master_train.columns and 'pet' in master_train.columns:
        master_train['ndmi_pet_ratio'] = master_train['NDMI'] / (master_train['pet'] + 1e-6)
        master_test['ndmi_pet_ratio'] = master_test['NDMI'] / (master_test['pet'] + 1e-6)
    
    # --- Soil-population interaction ---
    if 'soil_clay_mean_0_5cm' in master_train.columns and 'worldpop_mean_1km' in master_train.columns:
        master_train['clay_x_population'] = master_train['soil_clay_mean_0_5cm'] * master_train['worldpop_mean_1km']
        master_test['clay_x_population'] = master_test['soil_clay_mean_0_5cm'] * master_test['worldpop_mean_1km']
    
    # --- Basin dilution proxy ---
    if 'river_avg_discharge_cms' in master_train.columns and 'basin_upstream_area_km2' in master_train.columns:
        master_train['discharge_per_area'] = (
            master_train['river_avg_discharge_cms'] / (master_train['basin_upstream_area_km2'] + 1e-6)
        )
        master_test['discharge_per_area'] = (
            master_test['river_avg_discharge_cms'] / (master_test['basin_upstream_area_km2'] + 1e-6)
        )
    
    # --- Seasonal indicator dari tanggal ---
    # (Harus dijalankan SEBELUM drop Sample_Date)
    if 'Sample_Date' in master_train.columns:
        master_train['month'] = master_train['Sample_Date'].dt.month
        master_train['season'] = master_train['month'].map(
            {12: 0, 1: 0, 2: 0,  # Summer (SA)
             3: 1, 4: 1, 5: 1,   # Autumn
             6: 2, 7: 2, 8: 2,   # Winter
             9: 3, 10: 3, 11: 3} # Spring
        )
        # Sin/cos encoding untuk cyclical nature
        master_train['month_sin'] = np.sin(2 * np.pi * master_train['month'] / 12)
        master_train['month_cos'] = np.cos(2 * np.pi * master_train['month'] / 12)
    
    new_feat_count = len([c for c in master_train.columns if c not in non_feature])
    print(f"  ✓ Features: {feat_count} → {new_feat_count} (+{new_feat_count - feat_count})")
else:
    print("ℹ Feature Engineering TIDAK aktif (iterasi 1 baseline)")
    print("  Set ENABLE_FEATURE_ENGINEERING = True untuk iterasi 2")

In [ ]:
export_dataframe_to_stage(master_train, "final_master_train.parquet")
export_dataframe_to_stage(master_test, "final_master_test.parquet")

In [ ]:
def analyze_parquet_from_stage(session, file_name, stage_path=STAGE):
    local_path = f"/tmp/{file_name}"
    if not os.path.exists(local_path):
        session.file.get(f"{stage_path}/{file_name}", "/tmp/")
    df = pd.read_parquet(local_path)
    
    print(f"--- Analyze: {file_name} ---")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    print("Columns & dtypes:")
    print(df.info())
    
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = missing_pct[missing_pct > 0].reset_index()
    if missing_df.empty:
        print("\n✓ Clean data — 0% NaN")
    else:
        missing_df.columns = ['Feature', 'NaN %']
        missing_df = missing_df.sort_values('NaN %', ascending=False).reset_index(drop=True)
        print(f"\n⚠ Missing values:")
        print(missing_df)
    
    return df

# Jalankan:
analyze_parquet_from_stage(session, "master_train.parquet")

In [ ]:
analyze_parquet_from_stage(session, "master_test.parquet")